In [ ]:

import pandas as pd
import requests
import os

# API key privately
from getpass import getpass
API_KEY = getpass("Enter your Census API key: ")

all_data = []

for year in list(range(2015, 2020)) + list(range(2021, 2025)):
    url = f"https://api.census.gov/data/{year}/acs/acs1"

    params = {
        "get": "NAME,B19013_001E",
        "for": "state:*",
        "key": API_KEY
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data[1:], columns=data[0])
    df["year"] = year

    all_data.append(df)

    print(f"Downloaded {year}: {len(df)} rows")

income_data = pd.concat(all_data, ignore_index=True)

# Save raw data
income_data.to_csv("census_income_raw.csv", index=False)

print("Total rows:", len(income_data))
income_data.head()


In [ ]:

# structure of data
print("Dataset Information:")
income_data.info()

# missing values
print("\nMissing Values:")
print(income_data.isnull().sum())

# duplicate combinations
print("\nDuplicate State-Year Records:")
print(income_data.duplicated(subset=["state", "year"]).sum())

# years
print("\nYears Available:")
print(sorted(income_data["year"].unique()))

# income values 
print("\nIncome Summary:")
print(pd.to_numeric(
    income_data["B19013_001E"], errors="coerce"
).describe())


In [ ]:

# making copy in case
income_cleaned = income_data.copy()

# Rename columns to read easier
income_cleaned = income_cleaned.rename(columns={
    "NAME": "state_name",
    "B19013_001E": "median_household_income",
    "state": "state_fips"
})

# convert income to numeric
income_cleaned["median_household_income"] = pd.to_numeric(
    income_cleaned["median_household_income"],
    errors="coerce"
)

# checking for invalid Census values (negative or zero)
income_cleaned.loc[
    income_cleaned["median_household_income"] <= 0,
    "median_household_income"
] = float("nan")

# standardize the state names
income_cleaned["state_name"] = (
    income_cleaned["state_name"].str.strip()
)

# Keep FIPS codes as two-character strings
income_cleaned["state_fips"] = (
    income_cleaned["state_fips"].astype(str).str.zfill(2)
)

# geographic areas
print("Geographic areas:")
print(sorted(income_cleaned["state_name"].unique()))

# check any missing income values after cleaning
print("\nMissing income values:")
print(income_cleaned["median_household_income"].isna().sum())

# look for duplicates
print("\nDuplicate state-year records:")
print(income_cleaned.duplicated(
    subset=["state_fips", "year"]
).sum())

# saving the now cleaned data 
income_cleaned.to_csv("census_income_cleaned.csv", index=False)

print("\nCleaned dataset saved!")
print("Shape:", income_cleaned.shape)

income_cleaned.head()
